In [40]:
# @title Setup Env
%pip install -q transformers datasets accelerate bitsandbytes sentence-transformers chromadb \
                langchain langchain-huggingface langchain-chroma langchain-community langchain-ollama langchain_text_splitters\
                sentencepiece \
                --upgrade

Note: you may need to restart the kernel to use updated packages.


In [41]:
# @title Install PDF + eval support
%pip install -q PyPDF2 --upgrade

Note: you may need to restart the kernel to use updated packages.


# 2-stage IR — PT-BR native stack + Evaluation

Full Portuguese-native RAG pipeline with evaluation loop:
- **Stage 1**: Dense retrieval with Serafim 900M-IR (PORTULAN)
- **Stage 2**: Cross-encoder reranking with mDeBERTa mMARCO
- **Generation**: Gervásio 8B PT-PT (PORTULAN), pre-quantized 4-bit official release
- **Evaluation**: LLM-as-a-Judge + NDCG@K, MRR@K, Precision/Recall@K, chunk IoU

In [42]:
from __future__ import annotations
import langchain, langchain_core

from dataclasses import dataclass, field
from typing import List, Sequence, Optional, Dict, Any, Tuple

import gc
import json
import math
import re
import numpy as np
import torch
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from PyPDF2 import PdfReader

import os

langchain.__version__, langchain_core.__version__

('1.2.13', '1.2.20')

In [43]:
# ============================================================
# Configuration
# ============================================================

@dataclass
class RetrievalConfig:
    # Serafim 900M fine-tuned for information retrieval (PT-BR native)
    embedding_model_name: str = "PORTULAN/serafim-900m-portuguese-pt-sentence-encoder-ir"

    # Multilingual cross-encoder reranker trained on mMARCO (includes Portuguese)
    reranker_model_name: str = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

    # Gervásio 8B PT-PT — official pre-quantized 4-bit release (no BitsAndBytes needed)
    llm_model_name: str = "PORTULAN/gervasio-8b-portuguese-ptpt-decoder-quantized-4bit"

    # Chroma persistence
    persist_directory: str = "./chroma_langchain_db"
    collection_name: str = "basic_ir_collection"

    # Retrieval sizes
    initial_k: int = 30
    final_k: int = 6

    # Device and performance
    device: Optional[str] = None
    normalize_embeddings: bool = True
    reranker_batch_size: int = 16

    # LLM generation settings
    max_new_tokens: int = 512

    # Evaluation — K values to compute metrics at
    eval_k_values: List[int] = field(default_factory=lambda: [1, 3, 5, 6])

In [44]:
# ============================================================
# Utility functions
# ============================================================

def get_device(explicit_device: Optional[str] = None) -> str:
    if explicit_device:
        return explicit_device
    return "cuda" if torch.cuda.is_available() else "cpu"


def free_gpu_memory() -> None:
    """Force-release cached VRAM. Call between heavy model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        print(f"VRAM — allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB")


def print_ranked_results(
    docs: Sequence[Document],
    score_key: str = "reranker_score",
    max_chars: int = 400,
) -> None:
    if not docs:
        print("No documents found.")
        return
    for idx, doc in enumerate(docs, start=1):
        score  = doc.metadata.get(score_key, None)
        source = doc.metadata.get("source", "N/A")
        print("=" * 100)
        print(f"Rank: {idx}  |  Source: {source}")
        if score is not None:
            print(f"Reranker score: {score:.6f}")
        print("-" * 100)
        print(doc.page_content[:max_chars].strip())
        print()

In [45]:
# ============================================================
# Two-stage retriever
# ============================================================

class TwoStageRetriever:
    """
    Two-stage retrieval pipeline:
    1. Dense retrieval from Chroma using Serafim-IR embeddings
    2. Cross-encoder reranking using mDeBERTa mMARCO
    """

    def __init__(
        self,
        vectorstore: Chroma,
        reranker_model_name: str,
        initial_k: int = 30,
        final_k: int = 6,
        device: Optional[str] = None,
        reranker_batch_size: int = 16,
    ) -> None:
        self.vectorstore = vectorstore
        self.initial_k = initial_k
        self.final_k = final_k
        self.device = get_device(device)
        self.reranker_batch_size = reranker_batch_size

        print(f"Loading reranker on {self.device}...")
        self.cross_encoder = CrossEncoder(reranker_model_name, device=self.device)
        print("Reranker ready.")

    def _dense_retrieve(self, query: str) -> List[Document]:
        return self.vectorstore.similarity_search(query, k=self.initial_k)

    def _rerank(self, query: str, docs: Sequence[Document]) -> List[Document]:
        if not docs:
            return []
        pairs  = [(query, doc.page_content) for doc in docs]
        scores = self.cross_encoder.predict(
            pairs, batch_size=self.reranker_batch_size, show_progress_bar=False
        )
        reranked: List[Document] = []
        for doc, score in zip(docs, scores):
            meta = dict(doc.metadata) if doc.metadata else {}
            meta["reranker_score"] = float(score)
            reranked.append(Document(page_content=doc.page_content, metadata=meta))
        reranked.sort(key=lambda d: d.metadata["reranker_score"], reverse=True)
        return reranked[: self.final_k]

    def invoke(self, query: str) -> List[Document]:
        return self._rerank(query, self._dense_retrieve(query))

    def batch(self, queries: Sequence[str]) -> List[List[Document]]:
        return [self.invoke(q) for q in queries]

In [46]:
# ============================================================
# Build vector store (Serafim-IR embeddings)
# ============================================================

def build_embeddings(config: RetrievalConfig, force_cpu: bool = False) -> HuggingFaceEmbeddings:
    device = "cpu" if force_cpu else get_device(config.device)
    print(f"Loading Serafim embeddings on {device}...")
    return HuggingFaceEmbeddings(
        model_name=config.embedding_model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": config.normalize_embeddings},
    )


def build_vectorstore(config: RetrievalConfig, force_cpu: bool = False) -> Chroma:
    embeddings = build_embeddings(config, force_cpu=force_cpu)
    return Chroma(
        collection_name=config.collection_name,
        embedding_function=embeddings,
        persist_directory=config.persist_directory,
    )

In [47]:
# ============================================================
# PDF loading — improved chunking with overlap + IoU diagnostics
# ============================================================

def token_set(text: str) -> set:
    """Lowercase word-token set for IoU calculation."""
    return set(re.findall(r"\w+", text.lower()))


def chunk_iou(chunk_a: str, chunk_b: str) -> float:
    """
    Token-level Intersection over Union between two chunks.
    High IoU (>0.5) means chunks are too similar / overlap is too large.
    Low average IoU means chunks are well-separated.
    """
    a, b = token_set(chunk_a), token_set(chunk_b)
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def diagnose_chunks(texts: List[str], sample_size: int = 5) -> None:
    """
    Print chunk diagnostics:
    - Token count distribution
    - Average IoU between consecutive chunks (overlap health check)
    """
    token_counts = [len(token_set(t)) for t in texts]
    print(f"\n{'─'*60}")
    print(f"Chunk diagnostics ({len(texts)} chunks)")
    print(f"  Tokens/chunk — min: {min(token_counts)}  "
          f"max: {max(token_counts)}  "
          f"mean: {sum(token_counts)/len(token_counts):.1f}")

    # IoU between consecutive chunks — measures overlap bleed
    ious = [
        chunk_iou(texts[i], texts[i + 1])
        for i in range(min(len(texts) - 1, 200))  # cap at 200 pairs
    ]
    mean_iou = sum(ious) / len(ious) if ious else 0.0
    print(f"  Consecutive chunk IoU — mean: {mean_iou:.3f}  "
          f"max: {max(ious):.3f}")
    if mean_iou > 0.4:
        print("  ⚠ High IoU — consider reducing chunk_overlap or increasing chunk_size")
    elif mean_iou < 0.05:
        print("  ⚠ Very low IoU — chunks may be too large, context continuity could be lost")
    else:
        print("  ✓ IoU looks healthy")
    print(f"{'─'*60}\n")


def load_pdf_documents(
    pdf_path: str,
    chunk_size: int = 1200,
    chunk_overlap: int = 150,
    run_diagnostics: bool = True,
) -> Tuple[List[str], List[Dict[str, Any]]]:
    reader    = PdfReader(pdf_path)
    full_text = "".join(page.extract_text() or "" for page in reader.pages)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    chunks    = splitter.split_text(full_text)
    texts     = [c.strip() for c in chunks if c.strip()]
    metadatas = [{"source": pdf_path, "chunk": i} for i in range(len(texts))]

    if run_diagnostics:
        diagnose_chunks(texts)

    return texts, metadatas


def add_documents_to_vectorstore(
    vectorstore: Chroma,
    texts: Sequence[str],
    metadatas: Optional[Sequence[Dict[str, Any]]] = None,
) -> None:
    if metadatas is None:
        metadatas = [{} for _ in texts]
    docs = [Document(page_content=t, metadata=m) for t, m in zip(texts, metadatas)]
    vectorstore.add_documents(docs)

In [48]:
# ============================================================
# Build LLM — Gervásio 8B PT-PT, official pre-quantized 4-bit release
# Tokenizer loaded from base model — quantized repo is missing tokenizer files
# ============================================================

# Base model repo — used only for the tokenizer, not the weights
_GERVASIO_BASE = "PORTULAN/gervasio-8b-portuguese-ptpt-decoder"

def build_llm(config: RetrievalConfig) -> HuggingFacePipeline:
    print(f"Loading tokenizer from {_GERVASIO_BASE}...")
    tokenizer = AutoTokenizer.from_pretrained(_GERVASIO_BASE)
    print(f"Loading weights from {config.llm_model_name} (pre-quantized)...")
    model = AutoModelForCausalLM.from_pretrained(
        config.llm_model_name,
        device_map="auto",
    )
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=config.max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    print("LLM ready.")
    return HuggingFacePipeline(pipeline=pipe)

In [49]:
# ============================================================
# Build RAG chain
# ============================================================

def build_rag_chain(retriever: TwoStageRetriever, llm: HuggingFacePipeline):
    prompt = ChatPromptTemplate.from_template("""
Você é um assistente útil e preciso. Responda à pergunta abaixo com base apenas
no contexto fornecido. Se a resposta não estiver no contexto, diga que não sabe.
Não invente informações.

Contexto:
{context}

Pergunta: {question}

Resposta:""")

    def format_docs(docs: List[Document]) -> str:
        return "\n\n".join(
            f"[Trecho {i+1}]\n{doc.page_content}"
            for i, doc in enumerate(docs)
        )

    return (
        {
            "context":  RunnableLambda(retriever.invoke) | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )


def build_lcel_rerank_runnable(retriever: TwoStageRetriever):
    return RunnableLambda(retriever.invoke)

## Evaluation

Three complementary evaluation layers from the whiteboard:

1. **Retrieval metrics** (NDCG@K, MRR@K, Precision@K, Recall@K) — require a relevance-labelled test set
2. **Chunk diagnostics** (# tokens, IoU) — run automatically during PDF loading
3. **LLM-as-a-Judge** — uses Gervásio itself to score each answer on faithfulness and relevance, no labels needed

In [50]:
# ============================================================
# Retrieval metrics — NDCG@K, MRR@K, Precision@K, Recall@K
# ============================================================
# These require a test set: a list of (query, relevant_chunk_ids) pairs.
# relevant_chunk_ids should be the "chunk" values from the document metadata.
#
# Example:
#   test_set = [
#       {"query": "Qual a capacidade da bateria?", "relevant_ids": {12, 13}},
#       {"query": "Como fazer reset de fábrica?",  "relevant_ids": {47}},
#   ]
# ============================================================

def dcg_at_k(relevances: List[int], k: int) -> float:
    """Discounted Cumulative Gain at K."""
    relevances = relevances[:k]
    return sum(
        rel / math.log2(rank + 2)
        for rank, rel in enumerate(relevances)
    )


def ndcg_at_k(relevances: List[int], k: int) -> float:
    """Normalised DCG@K. Returns 1.0 for perfect ranking."""
    ideal = sorted(relevances, reverse=True)
    idcg  = dcg_at_k(ideal, k)
    return dcg_at_k(relevances, k) / idcg if idcg > 0 else 0.0


def mrr_at_k(relevances: List[int], k: int) -> float:
    """Mean Reciprocal Rank — position of the first relevant result."""
    for rank, rel in enumerate(relevances[:k], start=1):
        if rel > 0:
            return 1.0 / rank
    return 0.0


def precision_at_k(relevances: List[int], k: int) -> float:
    return sum(r > 0 for r in relevances[:k]) / k


def recall_at_k(relevances: List[int], k: int, total_relevant: int) -> float:
    if total_relevant == 0:
        return 0.0
    return sum(r > 0 for r in relevances[:k]) / total_relevant


def evaluate_retrieval(
    retriever: TwoStageRetriever,
    test_set: List[Dict[str, Any]],
    k_values: List[int] = [1, 3, 5, 6],
) -> Dict[str, float]:
    """
    Run all retrieval metrics over a labelled test set.

    Args:
        test_set: list of dicts with keys 'query' and 'relevant_ids' (set of chunk ints)
        k_values: list of K values to compute metrics at

    Returns:
        Dict of metric_name -> mean score across the test set
    """
    accumulators: Dict[str, List[float]] = {}

    for item in test_set:
        query        = item["query"]
        relevant_ids = set(item["relevant_ids"])

        docs = retriever.invoke(query)
        retrieved_ids = [doc.metadata.get("chunk", -1) for doc in docs]

        # Binary relevance list in retrieval order
        relevances = [1 if cid in relevant_ids else 0 for cid in retrieved_ids]

        for k in k_values:
            accumulators.setdefault(f"ndcg@{k}",      []).append(ndcg_at_k(relevances, k))
            accumulators.setdefault(f"mrr@{k}",       []).append(mrr_at_k(relevances, k))
            accumulators.setdefault(f"precision@{k}", []).append(precision_at_k(relevances, k))
            accumulators.setdefault(f"recall@{k}",    []).append(
                recall_at_k(relevances, k, len(relevant_ids))
            )

    results = {metric: float(np.mean(scores)) for metric, scores in accumulators.items()}

    print("\nRetrieval Metrics")
    print("─" * 50)
    for k in k_values:
        print(f"  @{k:2d}  NDCG: {results[f'ndcg@{k}']:.3f}  "
              f"MRR: {results[f'mrr@{k}']:.3f}  "
              f"P: {results[f'precision@{k}']:.3f}  "
              f"R: {results[f'recall@{k}']:.3f}")
    print("─" * 50)

    return results

In [51]:
# ============================================================
# LLM-as-a-Judge
# Scores each answer on two axes:
#   - Faithfulness (0-5): is the answer grounded in the context?
#   - Relevance   (0-5): does it actually answer the question?
# The same Gervásio 8B is reused — no second model needed.
# ============================================================

JUDGE_PROMPT = ChatPromptTemplate.from_template("""
Você é um avaliador rigoroso de sistemas de perguntas e respostas.
Avalie a resposta abaixo com base no contexto e na pergunta fornecidos.

Dê duas notas de 0 a 5:
- **Fidelidade**: a resposta é completamente baseada no contexto? (0 = inventa, 5 = totalmente fiel)
- **Relevância**: a resposta realmente responde à pergunta? (0 = não responde, 5 = responde completamente)

Responda APENAS com JSON válido, sem texto adicional, no formato:
{{"fidelidade": <0-5>, "relevancia": <0-5>, "justificativa": "<uma frase>"}}

Contexto:
{context}

Pergunta: {question}

Resposta avaliada: {answer}

JSON:"""
)


@dataclass
class JudgeResult:
    faithfulness:    float
    relevance:       float
    justification:   str
    raw_output:      str

    @property
    def score(self) -> float:
        """Combined score (simple average, both axes weighted equally)."""
        return (self.faithfulness + self.relevance) / 2

    def __str__(self) -> str:
        return (
            f"Fidelidade: {self.faithfulness}/5  "
            f"Relevância: {self.relevance}/5  "
            f"Score: {self.score:.1f}/5\n"
            f"Justificativa: {self.justification}"
        )


def parse_judge_output(raw: str) -> Tuple[float, float, str]:
    """Extract scores from the LLM judge JSON output robustly."""
    # Strip markdown fences if present
    raw = re.sub(r"```(?:json)?", "", raw).strip()
    # Find the first JSON object in the output
    match = re.search(r"\{.*?\}", raw, re.DOTALL)
    if not match:
        return 0.0, 0.0, f"Parse failed: {raw[:100]}"
    try:
        data          = json.loads(match.group())
        faithfulness  = float(data.get("fidelidade",    data.get("faithfulness", 0)))
        relevance     = float(data.get("relevancia",    data.get("relevance",    0)))
        justification = str(data.get("justificativa",   data.get("justification", "")))
        return faithfulness, relevance, justification
    except (json.JSONDecodeError, ValueError) as e:
        return 0.0, 0.0, f"Parse error: {e}"


def judge_answer(
    llm: HuggingFacePipeline,
    question: str,
    context: str,
    answer: str,
) -> JudgeResult:
    """
    Ask the LLM to score a question/context/answer triple.
    Returns a JudgeResult with faithfulness, relevance, and justification.
    """
    judge_chain = JUDGE_PROMPT | llm | StrOutputParser()
    raw = judge_chain.invoke({
        "context":  context,
        "question": question,
        "answer":   answer,
    })
    faithfulness, relevance, justification = parse_judge_output(raw)
    return JudgeResult(
        faithfulness=faithfulness,
        relevance=relevance,
        justification=justification,
        raw_output=raw,
    )


def evaluate_rag(
    retriever: TwoStageRetriever,
    llm: HuggingFacePipeline,
    rag_chain,
    questions: List[str],
) -> List[Dict[str, Any]]:
    """
    Run LLM-as-a-Judge over a list of questions.
    For each question: retrieves context, generates answer, judges it.
    Returns a list of result dicts and prints a summary table.
    """
    results = []

    for i, question in enumerate(questions, 1):
        print(f"Evaluating {i}/{len(questions)}: {question[:60]}...")

        docs    = retriever.invoke(question)
        context = "\n\n".join(
            f"[Trecho {j+1}]\n{d.page_content}" for j, d in enumerate(docs)
        )
        answer  = rag_chain.invoke(question)
        verdict = judge_answer(llm, question, context, answer)

        results.append({
            "question":     question,
            "answer":       answer,
            "faithfulness": verdict.faithfulness,
            "relevance":    verdict.relevance,
            "score":        verdict.score,
            "justification": verdict.justification,
        })

    # Summary table
    mean_f = np.mean([r["faithfulness"] for r in results])
    mean_r = np.mean([r["relevance"]    for r in results])
    mean_s = np.mean([r["score"]        for r in results])

    print("\n" + "=" * 80)
    print("LLM-as-a-Judge Summary")
    print("=" * 80)
    print(f"{'Pergunta':<45} {'Fid':>5} {'Rel':>5} {'Score':>7}")
    print("─" * 80)
    for r in results:
        q = r["question"][:43] + ".." if len(r["question"]) > 45 else r["question"]
        print(f"{q:<45} {r['faithfulness']:>5.1f} {r['relevance']:>5.1f} {r['score']:>7.2f}")
    print("─" * 80)
    print(f"{'MÉDIA':<45} {mean_f:>5.1f} {mean_r:>5.1f} {mean_s:>7.2f}")
    print("=" * 80)

    return results

In [ ]:
# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    config = RetrievalConfig(
        embedding_model_name  = "PORTULAN/serafim-900m-portuguese-pt-sentence-encoder-ir",
        reranker_model_name   = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
        llm_model_name        = "PORTULAN/gervasio-8b-portuguese-ptpt-decoder-quantized-4bit",
        persist_directory     = "./chroma_langchain_db",
        collection_name       = "basic_ir_collection",
        initial_k             = 30,
        final_k               = 6,
        device                = None,
        normalize_embeddings  = True,
        reranker_batch_size   = 16,
        max_new_tokens        = 512,
        eval_k_values         = [1, 3, 5, 6],
    )

    print(f"Using device: {get_device(config.device)}")

    # ── Step 1: Index with Serafim on GPU ─────────────────────────────────────
    vectorstore = build_vectorstore(config, force_cpu=False)

    pdf_path = "../PDFs/SM-A15X_A16X_A17X_A06X_A075_16_Emb_BR_Rev.2.1.pdf"

    if os.path.exists(pdf_path):
        print(f"Loading {pdf_path}...")
        sample_texts, sample_metadatas = load_pdf_documents(
            pdf_path, chunk_size=1200, chunk_overlap=150, run_diagnostics=True
        )
        print(f"Loaded {len(sample_texts)} chunks")
    else:
        print(f"PDF not found. Using sample texts.")
        sample_texts = [
            "Autoencoders são redes neurais treinadas para reconstruir dados de entrada.",
            "U-Nets são arquiteturas encoder-decoder amplamente usadas em segmentação de imagens.",
            "Cross-encoders avaliam pares query-documento juntos e são usados para reranking.",
            "Bi-encoders codificam queries e documentos independentemente em um espaço vetorial compartilhado.",
            "Recuperação em dois estágios combina busca densa rápida com reranking mais preciso.",
        ]
        sample_metadatas = [{"source": "sample", "chunk": i} for i in range(len(sample_texts))]

    add_documents_to_vectorstore(vectorstore, sample_texts, sample_metadatas)

    # ── Step 2: Free GPU memory from embedding model ──────────────────────────
    embedding_fn = vectorstore._embedding_function
    if hasattr(embedding_fn, "client"):
        del embedding_fn.client
    del embedding_fn
    free_gpu_memory()

    vectorstore = build_vectorstore(config, force_cpu=True)

    # ── Step 3: Load reranker + LLM ───────────────────────────────────────────
    retriever = TwoStageRetriever(
        vectorstore         = vectorstore,
        reranker_model_name = config.reranker_model_name,
        initial_k           = config.initial_k,
        final_k             = config.final_k,
        device              = config.device,
        reranker_batch_size = config.reranker_batch_size,
    )

    llm = build_llm(config)
    free_gpu_memory()

    rag_chain = build_rag_chain(retriever, llm)

    # ── Step 4: Single query ──────────────────────────────────────────────────
    query  = "Quais são as principais características e especificações?"
    answer = rag_chain.invoke(query)

    print("\n" + "=" * 100)
    print(f"Pergunta: {query}")
    print("-" * 100)
    print(f"Resposta: {answer}")
    print("=" * 100)

    # ── Step 5: LLM-as-a-Judge evaluation ────────────────────────────────────
    # Add your own questions here to evaluate the pipeline end-to-end
    eval_questions = [
        "Quais são as principais características e especificações?",
        "Como fazer reset de fábrica no dispositivo?",
        "Qual a capacidade da bateria e autonomia?",
    ]

    judge_results = evaluate_rag(retriever, llm, rag_chain, eval_questions)

    # ── Step 6: Retrieval metrics (requires labelled test set) ─────────────────
    # Uncomment and fill in relevant chunk IDs from your PDF to use this.
    # You can find chunk IDs by inspecting doc.metadata["chunk"] in retrieval results.
    #
    # labelled_test_set = [
    #     {"query": "Qual a capacidade da bateria?",       "relevant_ids": {12, 13}},
    #     {"query": "Como fazer reset de fábrica?",        "relevant_ids": {47}},
    #     {"query": "Quais câmeras o dispositivo possui?", "relevant_ids": {5, 6, 7}},
    # ]
    # retrieval_metrics = evaluate_retrieval(
    #     retriever, labelled_test_set, k_values=config.eval_k_values
    # )

Using device: cuda
Loading Serafim embeddings on cuda...


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 8104.22it/s]


Loading ../PDFs/SM-A15X_A16X_A17X_A06X_A075_16_Emb_BR_Rev.2.1.pdf...

────────────────────────────────────────────────────────────
Chunk diagnostics (169 chunks)
  Tokens/chunk — min: 56  max: 124  mean: 95.4
  Consecutive chunk IoU — mean: 0.257  max: 0.444
  ✓ IoU looks healthy
────────────────────────────────────────────────────────────

Loaded 169 chunks
VRAM — allocated: 4.02 GB | reserved: 4.13 GB
Loading Serafim embeddings on cpu...


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 8082.85it/s]


Loading reranker on cuda...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7058.34it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker ready.
Loading tokenizer from PORTULAN/gervasio-8b-portuguese-ptpt-decoder...
Loading weights from PORTULAN/gervasio-8b-portuguese-ptpt-decoder-quantized-4bit (pre-quantized)...


ValueError: Unrecognized model in PORTULAN/gervasio-8b-portuguese-ptpt-decoder-quantized-4bit. Should have a `model_type` key in its config.json.